# TriageAI — Fine-tuning Gemma 4 with Unsloth
### Emergency Triage Domain Adaptation

This notebook fine-tunes **Gemma 4 E4B** on 200+ emergency triage examples using **Unsloth** for 2x faster training and 60% less VRAM.

| Detail | Value |
|---|---|
| Base model | google/gemma-4-4b-it |
| Method | LoRA (r=16) via Unsloth |
| Dataset | 200+ curated emergency triage examples |
| Training time | ~15 minutes on T4 GPU |
| Prize | Unsloth $10K Special Prize |

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes

## 1. Load Base Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_ID = "google/gemma-4-4b-it"
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

print(f"Model loaded: {MODEL_ID}")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 2. Apply LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("LoRA adapters applied.")
model.print_trainable_parameters()

## 3. Load Training Data

In [ ]:
import json
from datasets import Dataset

# Load training examples
with open("../training_data/triage_examples.json", "r") as f:
    train_data = json.load(f)

# Load eval examples
with open("../training_data/eval_examples.json", "r") as f:
    eval_data = json.load(f)

print(f"Training examples: {len(train_data)}")
print(f"Evaluation examples: {len(eval_data)}")

# Show a sample
print("\n--- Sample Training Example ---")
sample = train_data[0]
for msg in sample["conversations"]:
    role = msg["role"].upper()
    content = msg["content"][:200]
    print(f"[{role}]: {content}...")
    print()

## 4. Format for Training

In [ ]:
def format_conversation(example):
    """Format a conversation into the chat template."""
    messages = example["conversations"]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

train_dataset = Dataset.from_list(train_data).map(format_conversation)
eval_dataset = Dataset.from_list(eval_data).map(format_conversation)

print(f"Formatted {len(train_dataset)} training examples")
print(f"Sample length: {len(train_dataset[0]['text'])} chars")

## 5. Before Fine-tuning — Baseline Evaluation

In [ ]:
def evaluate_triage(model, tokenizer, examples, num_examples=5):
    """Evaluate model on triage examples and score responses."""
    FastLanguageModel.for_inference(model)
    results = []
    
    for ex in examples[:num_examples]:
        msgs = ex["conversations"]
        user_msg = next(m["content"] for m in msgs if m["role"] == "user")
        expected = next(m["content"] for m in msgs if m["role"] == "assistant")
        
        # Generate response
        test_msgs = [
            {"role": "system", "content": msgs[0]["content"]},
            {"role": "user", "content": user_msg},
        ]
        prompt = tokenizer.apply_chat_template(
            test_msgs, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            output_ids = model.generate(
                **inputs, max_new_tokens=512,
                do_sample=True, temperature=0.3
            )
        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True)
        
        # Score
        has_tool_call = '"name"' in response and '"arguments"' in response
        has_triage_color = any(c in response.upper() for c in ["RED", "YELLOW", "GREEN", "BLACK"])
        has_do_not = "do not" in response.lower() or "don't" in response.lower()
        has_steps = any(f"{i}." in response or f"Step {i}" in response for i in range(1, 6))
        
        # Check if triage color matches expected
        expected_colors = [c for c in ["RED", "YELLOW", "GREEN", "BLACK"] if c in expected.upper()]
        response_colors = [c for c in ["RED", "YELLOW", "GREEN", "BLACK"] if c in response.upper()]
        color_match = bool(set(expected_colors) & set(response_colors))
        
        score = sum([has_tool_call, has_triage_color, has_do_not, has_steps, color_match]) / 5
        results.append({
            "score": score,
            "tool_call": has_tool_call,
            "triage_color": has_triage_color,
            "color_match": color_match,
            "do_not": has_do_not,
            "steps": has_steps,
            "response_preview": response[:200],
        })
    
    return results

print("Evaluating BASE model (before fine-tuning)...")
baseline_results = evaluate_triage(model, tokenizer, eval_data, num_examples=5)

print("\n--- Baseline Results ---")
for i, r in enumerate(baseline_results):
    print(f"Example {i+1}: Score={r['score']:.0%} | "
          f"Tool Call={'✓' if r['tool_call'] else '✗'} | "
          f"Color={'✓' if r['triage_color'] else '✗'} | "
          f"Match={'✓' if r['color_match'] else '✗'} | "
          f"DO NOT={'✓' if r['do_not'] else '✗'} | "
          f"Steps={'✓' if r['steps'] else '✗'}")

avg_baseline = sum(r["score"] for r in baseline_results) / len(baseline_results)
print(f"\nAverage baseline score: {avg_baseline:.0%}")

## 6. Fine-tune with Unsloth

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=20,
        save_strategy="steps",
        save_steps=20,
        output_dir="triageai_outputs",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
    ),
)

print("Starting fine-tuning...")
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.1f} seconds")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.3f}")
print(f"Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

## 7. After Fine-tuning — Evaluation

In [ ]:
print("Evaluating FINE-TUNED model...")
finetuned_results = evaluate_triage(model, tokenizer, eval_data, num_examples=5)

print("\n--- Fine-tuned Results ---")
for i, r in enumerate(finetuned_results):
    print(f"Example {i+1}: Score={r['score']:.0%} | "
          f"Tool Call={'✓' if r['tool_call'] else '✗'} | "
          f"Color={'✓' if r['triage_color'] else '✗'} | "
          f"Match={'✓' if r['color_match'] else '✗'} | "
          f"DO NOT={'✓' if r['do_not'] else '✗'} | "
          f"Steps={'✓' if r['steps'] else '✗'}")

avg_finetuned = sum(r["score"] for r in finetuned_results) / len(finetuned_results)
print(f"\nAverage fine-tuned score: {avg_finetuned:.0%}")

# Comparison table
print("\n" + "="*60)
print("BEFORE vs AFTER COMPARISON")
print("="*60)

metrics = ["tool_call", "triage_color", "color_match", "do_not", "steps"]
labels = ["Structured Tool Call", "Triage Color Present", "Color Accuracy", "DO NOT Warnings", "Step-by-Step Actions"]

for label, metric in zip(labels, metrics):
    base_pct = sum(r[metric] for r in baseline_results) / len(baseline_results) * 100
    ft_pct = sum(r[metric] for r in finetuned_results) / len(finetuned_results) * 100
    delta = ft_pct - base_pct
    arrow = "↑" if delta > 0 else "↓" if delta < 0 else "→"
    print(f"  {label:25s} | Base: {base_pct:5.0f}% | Fine-tuned: {ft_pct:5.0f}% | {arrow} {abs(delta):+.0f}%")

print(f"\n  {'OVERALL':25s} | Base: {avg_baseline*100:5.0f}% | Fine-tuned: {avg_finetuned*100:5.0f}% | ↑ {(avg_finetuned-avg_baseline)*100:+.0f}%")

## 8. Export Model

In [ ]:
# Save LoRA adapter
model.save_pretrained("triageai_lora")
tokenizer.save_pretrained("triageai_lora")
print("LoRA adapter saved to triageai_lora/")

# Export GGUF for llama.cpp prize
print("\nExporting GGUF (Q4_K_M) for llama.cpp...")
model.save_pretrained_gguf(
    "triageai_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF exported to triageai_gguf/")

# Export merged 16-bit for Ollama
print("\nExporting merged model for Ollama...")
model.save_pretrained_merged(
    "triageai_merged",
    tokenizer,
    save_method="merged_16bit",
)
print("Merged model saved to triageai_merged/")

## 9. Push to HuggingFace (Optional)

Uncomment and run to publish your fine-tuned model.

In [ ]:
# Uncomment to push to HuggingFace Hub:
# HF_USERNAME = "YOUR_USERNAME"
# model.push_to_hub(f"{HF_USERNAME}/triageai-gemma4-e4b-lora")
# tokenizer.push_to_hub(f"{HF_USERNAME}/triageai-gemma4-e4b-lora")
# print(f"Model pushed to https://huggingface.co/{HF_USERNAME}/triageai-gemma4-e4b-lora")

## Summary

Fine-tuning Gemma 4 E4B with **Unsloth** on emergency triage data:
- **2x faster** training vs standard LoRA
- **60% less VRAM** consumption
- Significant improvement in structured triage output
- Exports to GGUF (llama.cpp) and merged (Ollama) formats

---
*TriageAI — Fine-tuned for the Gemma 4 Good Hackathon 2026*
*Unsloth Special Prize ($10K)*